In [22]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [23]:
# loads all JSON data for each run in a dictionary
data_master = {}
for seed in range(0,10):
    data_list = {}
    path = f"results/final_run/sim_results_G1-40_T0_4_S{seed}.json"
    with open(path, 'r') as f:
        data = json.load(f)
        data_list.update(data)
    data_master[seed] = data_list

In [24]:
# loads all participant data from the original data and lists of groups and questions
with open('simulation_resources/participant_attitudes.json', 'r') as f:
    participants = json.load(f)

groups = list(set([int(x.split("_")[1]) for x in data_master[0].keys()]))
questions = pd.read_csv("AI1R_questions.csv")
# zip variable and variable_text columns into a dictionary
question_dict = dict(zip(questions['Variable'], questions['Variable Label']))
for key in question_dict.keys():
    question_dict[key] = question_dict[key].replace("[","").replace("]","")

AI1R_Data = pd.read_csv("clean_data/question_level_master_dataset.csv")

In [25]:
master_list = []
for group in groups:
    participant_ids = data_master[0][f"Group_{group}_Q1"]["rankings"].keys()
    for question in question_dict.keys():
        for participant_id in participant_ids:
            row_info = [int(participant_id), group, question]
            for run_id in range(0,10):
                LLM_response = data_master[run_id][f"Group_{group}_{question}"]["rankings"][participant_id]
                row_info.append(LLM_response)
            master_list.append(row_info)

master_list_df = pd.DataFrame(master_list, columns=["participant_id", "group", "question"] + [f"run_{i}" for i in range(0,10)])

In [31]:
experiment_df = AI1R_Data.merge(master_list_df, how='right', left_on=['ID', 'Question'], right_on=['participant_id', 'question'])

In [32]:
experiment_df.dtypes

ID                 int64
Question          object
QuestionText      object
Topic             object
ValidResponse       bool
BeforeNumber       int64
BeforeResponse    object
AfterNumber        int64
AfterResponse     object
GROUP              int64
RACE              object
AGEBRACKET        object
EDUCATION         object
PARTYBEFORE       object
GENDER            object
AGE                int64
participant_id     int64
group              int64
question          object
run_0              int64
run_1              int64
run_2              int64
run_3              int64
run_4              int64
run_5              int64
run_6              int64
run_7              int64
run_8              int64
run_9              int64
dtype: object

In [33]:
experiment_df.drop(columns=['participant_id', 'question','group', ], inplace=True)

In [35]:
experiment_df.to_csv("master_df_final.csv", index=False)